In [1]:
import subprocess, sys, os

# ── Paths ──────────────────────────────────────────────────────
RDT2_DIR           = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
OUTPUT_DIR         = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned"
DATASET_CONFIG_REL = "configs/datasets/custom_dataset.yaml"
LOG_PATH           = "/home/rishabh/Downloads/umi-pipeline-training/train_resume.log"
CHECKPOINT_1000    = os.path.join(OUTPUT_DIR, "checkpoint-1000")

os.chdir(RDT2_DIR)

os.environ['ACCELERATE_USE_DEEPSPEED']          = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION']  = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']           = 'expandable_segments:True'

# Verify checkpoint exists
if not os.path.exists(CHECKPOINT_1000):
    print(f"❌ Checkpoint not found: {CHECKPOINT_1000}")
else:
    print(f"✅ Resuming from: {CHECKPOINT_1000}")

cmd = [
    sys.executable, 'main.py',
    '--tokenizer_name',               'Qwen/Qwen2.5-VL-7B-Instruct',
    '--vae_name',                      'robotics-diffusion-transformer/RVQActionTokenizer',
    '--pretrained_model_name_or_path', 'robotics-diffusion-transformer/RDT2-VQ',
    '--output_dir',                    OUTPUT_DIR,
    '--resume_from_checkpoint',        CHECKPOINT_1000,   # ← resume here
    '--train_batch_size',              '1',
    '--gradient_accumulation_steps',   '8',
    '--eval_batch_size',               '1',
    '--max_train_steps',               '5000',            # ← total steps (runs 1000→5000)
    '--eval_strategy',                 'no',
    '--logging_steps',                 '10',
    '--checkpoints_total_limit',       '3',
    '--checkpointing_step',            '500',
    '--lr_scheduler',                  'cosine',
    '--learning_rate',                 '1e-6',
    '--mixed_precision',               'bf16',
    '--dataloader_num_workers',        '2',
    '--gradient_checkpointing',
    '--log_level',                     'info',
    '--report_to',                     'none',
    '--lr_warmup_steps',               '0',               # ← no warmup (already warmed up)
    '--dataset',                       DATASET_CONFIG_REL,
    '--image_corruption',
    '--use_default_collate_fn_for_eval',
    '--use_qlora',
    '--lora_r',                        '16',
    '--lora_alpha',                    '16',
    '--lora_dropout',                  '0.1',
]

print("🚀 Resuming training from step 1000 → 5000")
print(f"   LR: 1e-6  |  steps: 1000→5000  |  warmup: 0")
print(f"   Log: {LOG_PATH}")
print("=" * 70)

with open(LOG_PATH, 'w') as logf:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy()
    )
    for line in process.stdout:
        logf.write(line)
        logf.flush()
        print(line, end='', flush=True)

process.wait()
print("\n✅ Done!" if process.returncode == 0 else f"\n❌ Exit code {process.returncode}")

✅ Resuming from: /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000
🚀 Resuming training from step 1000 → 5000
   LR: 1e-6  |  steps: 1000→5000  |  warmup: 0
   Log: /home/rishabh/Downloads/umi-pipeline-training/train_resume.log

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.77s/it]
Trainable parameters: 49239808
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Loading model from /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-1000.
***** Running training *****
  Num examples = 40,000
  Num Epochs = 9,223,372,036,854

In [4]:
import os, sys, numpy as np
from PIL import Image

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
CHECKPOINT   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-5000"
NORMALIZER   = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
INSTRUCTION  = "pick up the marker and place it in the box"

sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Add this line

print("✅ Setup done")
print(f"   Checkpoint: {CHECKPOINT}")

✅ Setup done
   Checkpoint: /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-5000


In [5]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from models.normalizer.normalizer import LinearNormalizer
from models.multivqvae import MultiVQVAE

print("[1/4] Loading base model...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="cuda"
)

print("[2/4] Loading checkpoint-5000 adapter...")
model = PeftModel.from_pretrained(base, CHECKPOINT)
model.eval()
print(f"      ✅ {CHECKPOINT}")

print("[3/4] Loading processor...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
)

print("[4/4] Loading VAE...")
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer"
).cuda().eval()

print("\n✅ ALL MODELS LOADED!")

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/4] Loading base model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.45it/s]


[2/4] Loading checkpoint-5000 adapter...
      ✅ /home/rishabh/Downloads/umi-pipeline-training/outputs/rdt2-finetuned/checkpoint-5000
[3/4] Loading processor...
[4/4] Loading VAE...

✅ ALL MODELS LOADED!


In [6]:
import zarr, numpy as np
from PIL import Image

root = zarr.open(DATASET_PATH, mode='r')

# Load action stats from real dataset
pos  = np.array(root['data']['robot0_eef_pos'])
rot  = np.array(root['data']['robot0_eef_rot_axis_angle'])
grip = np.array(root['data']['robot0_gripper_width'])
actions = np.concatenate([pos, rot, grip], axis=-1)  # (78018, 7)

ACTION_MEAN = actions.mean(axis=0)
ACTION_STD  = actions.std(axis=0)
ACTION_STD  = np.where(ACTION_STD < 1e-8, 1.0, ACTION_STD)
ACTION_MIN  = actions.min(axis=0)
ACTION_MAX  = actions.max(axis=0)

print(f"✅ Dataset stats loaded")
print(f"   mean: {np.round(ACTION_MEAN, 4)}")
print(f"   std:  {np.round(ACTION_STD,  4)}")

# Load test image
img_arr = np.array(root['data']['camera0_rgb'][500], dtype=np.uint8)
img     = Image.fromarray(img_arr).resize((768, 384))
print(f"✅ Image loaded: {img_arr.shape} → 768x384")

✅ Dataset stats loaded
   mean: [ 0.0077 -0.1062  0.1275 -2.1636 -0.1622  0.0445  0.0265]
   std:  [0.157  0.1385 0.104  1.3838 0.8236 0.1868 0.0119]
✅ Image loaded: (224, 224, 3) → 768x384


In [7]:
import torch

print(f"🧠 Running inference...")
print(f"   Instruction: '{INSTRUCTION}'")

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": img},
        {"type": "text",  "text": INSTRUCTION}
    ]
}]

text   = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = processor(
    text=[text], images=[img], return_tensors="pt"
).to("cuda", torch.bfloat16)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
    )

input_len  = inputs["input_ids"].shape[1]
generated  = output_ids[:, input_len:].cpu()
token_ids  = generated[0].numpy()

del inputs
torch.cuda.empty_cache()

print(f"✅ Generated {len(token_ids)} tokens")
print(f"   IDs: {token_ids.tolist()}")

🧠 Running inference...
   Instruction: 'pick up the marker and place it in the box'


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


✅ Generated 30 tokens
   IDs: [151650, 151032, 150821, 150630, 150807, 151174, 151262, 150778, 151402, 151359, 151298, 150830, 151308, 151619, 151072, 151051, 151035, 151188, 150764, 150938, 151295, 151269, 150719, 151117, 151327, 150853, 151627, 150735, 151651, 151645]


In [8]:
import numpy as np, torch

# ── Remove special tokens ──────────────────────────────────────
SPECIAL = {151643, 151644, 151645, 151646, 151650, 151651,
           151652, 151653, 151654, 151655, 151656}
clean   = [t for t in token_ids.tolist() if t not in SPECIAL]
print(f"Clean tokens ({len(clean)}): {clean[:10]}...")

# ── VAE decode ────────────────────────────────────────────────
BASE     = 151000
pos_len  = vae.pos_id_len                    # 18
rot_len  = vae.rot_id_len                    # 6
grip_len = getattr(vae, 'grip_id_len', 3)    # 3
total    = pos_len + rot_len + grip_len       # 27

remapped = [max(0, t - BASE) for t in clean[:total]]
ids_t    = torch.tensor(remapped, dtype=torch.long).unsqueeze(0)

print(f"VAE input shape: {ids_t.shape}")
print(f"Token range: [{ids_t.min()}, {ids_t.max()}]")

vae_cpu = vae.cpu()
with torch.no_grad():
    result = vae_cpu.decode(ids_t)

print(f"VAE output shape: {result.shape}")  # (1, 24, 20)

# Take first timestep, first 7 dims
raw = result[0, 0].float().numpy()[:7]
print(f"Raw VAE output: {np.round(raw, 4)}")

# ── Unnormalize with dataset stats ────────────────────────────
a        = raw * ACTION_STD + ACTION_MEAN
a        = np.clip(a, ACTION_MIN, ACTION_MAX)

print(f"Unnormalized:   {np.round(a, 4)}")

# ── Convert to M750 format ────────────────────────────────────
m750 = [
    round(float(a[0]) * 1000, 1),
    round(float(a[1]) * 1000, 1),
    round(float(a[2]) * 1000, 1),
    round(float(np.degrees(a[3])), 1),
    round(float(np.degrees(a[4])), 1),
    round(float(np.degrees(a[5])), 1),
]
grip_pct = int(np.clip(
    (float(a[6]) - float(ACTION_MIN[6])) /
    (float(ACTION_MAX[6]) - float(ACTION_MIN[6]) + 1e-8) * 100,
    0, 100
))

print("\n" + "=" * 55)
print("✅ FINAL ACTION:")
print("=" * 55)
print(f"  x:       {a[0]:.4f} m  → {a[0]*1000:.1f} mm")
print(f"  y:       {a[1]:.4f} m  → {a[1]*1000:.1f} mm")
print(f"  z:       {a[2]:.4f} m  → {a[2]*1000:.1f} mm")
print(f"  rx:      {np.degrees(a[3]):.2f} deg")
print(f"  ry:      {np.degrees(a[4]):.2f} deg")
print(f"  rz:      {np.degrees(a[5]):.2f} deg")
print(f"  gripper: {a[6]:.4f} → {grip_pct}/100")
print("=" * 55)
print(f"\n🤖 M750 COMMAND:")
print(f"   send_coords({m750}, speed=15, mode=1)")
print(f"   set_gripper_value({grip_pct}, speed=50)")
print("=" * 55)

# Save for robot cell
last_m750 = m750
last_grip = grip_pct
print("\n✅ Ready for robot!")

Clean tokens (27): [151032, 150821, 150630, 150807, 151174, 151262, 150778, 151402, 151359, 151298]...
VAE input shape: torch.Size([1, 27])
Token range: [0, 627]
VAE output shape: torch.Size([1, 24, 20])
Raw VAE output: [ 1.280000e-02 -2.177000e-01 -9.900000e-02  3.539427e+02  5.617600e+00
  9.329500e+00 -2.248360e+01]
Unnormalized:   [ 0.0098 -0.1364  0.1172  3.1409  3.1406  1.1756  0.0065]

✅ FINAL ACTION:
  x:       0.0098 m  → 9.8 mm
  y:       -0.1364 m  → -136.4 mm
  z:       0.1172 m  → 117.2 mm
  rx:      179.96 deg
  ry:      179.94 deg
  rz:      67.36 deg
  gripper: 0.0065 → 0/100

🤖 M750 COMMAND:
   send_coords([9.8, -136.4, 117.2, 180.0, 179.9, 67.4], speed=15, mode=1)
   set_gripper_value(0, speed=50)

✅ Ready for robot!


In [6]:
!pip install pymycobot


In [7]:
# Emergency diagnostic - try different M750 commands
import time
from pymycobot import MyArm
arm = MyArm('/dev/ttyACM1', 115200)

# Method 1: Check connection
print(arm.is_controller_connected())

# Method 2: Release and re-enable servos
arm.release_all_servos()
time.sleep(2)
arm.power_on()
time.sleep(2)

# Method 3: Try setting speed differently

arm.send_coords([200, 0, 300, 0, 0, 0], speed=30)

-1


-1

In [ ]:
# CELL 6 — M750 Inference using RDT2-style approach

import time, numpy as np, torch
from PIL import Image
import cv2

PORT = '/dev/ttyACM1'
SPEED = 30
N_STEPS = 20
SLEEP = 2.5
USE_CAMERA = False  # Set to True when camera is working]
INSTRUCTION="PICK THE MARKER PLACE IN THE BOX"

def predict_action_rdt2_style(image, instruction):
    """
    RDT2-style inference adapted for single-arm M750
    Returns action chunk (24, 7) but we'll use only first timestep
    """
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    
    # Prepare image - RDT2 uses 384x384 for each camera
    # For single arm, we just use one camera view
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Resize to RDT2's expected size
    image = image.resize((384, 384))
    
    # Create messages in RDT2 format
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": instruction}
        ]
    }]
    
    # Apply chat template
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ) CELL 6 — M750 Inference using RDT2-style approach

import time, numpy as np, torch
from PIL import Image
import cv2

PORT = '/dev/ttyACM1'
SPEED = 30
N_STEPS = 20
SLEEP = 2.5
USE_CAMERA = False  # Set to True when camera is working]
INSTRUCTION="PICK THE MARKER PLACE IN THE BOX"

def predict_action_rdt2_style(image, instruction):
    """
    RDT2-style inference adapted for single-arm M750
    Returns action chunk (24, 7) but we'll use only first timestep
    """
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    
    # Prepare image - RDT2 uses 384x384 for each camera
    # For single arm, we just use one camera view
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Resize to RDT2's expected size
    image = image.resize((384, 384))
    
    # Create messages in RDT2 format
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": instruction}
        ]
    }]
    
    # Apply chat template
    text = processor.apply
    
    # Process inputs
    inputs = processor(
        text=[text], 
        images=[image], 
        return_tensors="pt"
    ).to("cuda", torch.bfloat16)
    
    # Generate tokens
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )
    
    # Extract generated tokens
    input_len = inputs["input_ids"].shape[1]
    generated = output_ids[:, input_len:].cpu()
    token_ids = generated[0].numpy()
    
    del inputs
    torch.cuda.empty_cache()
    
    # Remove special tokens
    SPECIAL = {151643, 151644, 151645, 151646, 151650, 151651,
               151652, 151653, 151654, 151655, 151656}
    clean = [t for t in token_ids.tolist() if t not in SPECIAL]
    
    # Decode with VAE
    BASE = 151000
    pos_len = vae.pos_id_len    # 18
    rot_len = vae.rot_id_len    # 6
    grip_len = getattr(vae, 'grip_id_len', 3)  # 3
    total = pos_len + rot_len + grip_len  # 27
    
    remapped = [max(0, t - BASE) for t in clean[:total]]
    ids_t = torch.tensor(remapped, dtype=torch.long).unsqueeze(0)
    
    vae_cpu = vae.cpu()
    with torch.no_grad():
        result = vae_cpu.decode(ids_t)  # (1, 24, 20)
    
    # RDT2 outputs (24, 20) but we trained on single-arm data
    # So we'll extract only the right arm actions (first 10 dims) or all if single-arm
    action_chunk = result[0].float().numpy()  # (24, 20)
    
    # For M750 single arm, take first 10 dimensions (one arm's action)
    # [0-2]: position (x, y, z)
    # [3-8]: rotation (6D representation) 
    # [9]: gripper
    single_arm_chunk = action_chunk[:, :10]  # (24, 10)
    
    return single_arm_chunk


def convert_to_m750_format(action_chunk_step):
    """
    Convert RDT2 action format to M750 format
    Input: (10,) array with [x, y, z, rot_6d(6), gripper]
    Output: M750 coords [x_mm, y_mm, z_mm, rx_deg, ry_deg, rz_deg], gripper_pct
    """
    # Unnormalize
    a = action_chunk_step[:7] * ACTION_STD[:7] + ACTION_MEAN[:7]
    a = np.clip(a, ACTION_MIN[:7], ACTION_MAX[:7])
    
    # Convert to M750 format
    m750 = [
        round(float(a[0]) * 1000, 1),  # m to mm
        round(float(a[1]) * 1000, 1),
        round(float(a[2]) * 1000, 1),
        round(float(np.degrees(a[3])), 1),  # rad to deg
        round(float(np.degrees(a[4])), 1),
        round(float(np.degrees(a[5])), 1),
    ]
    
    # Gripper
    grip_pct = int(np.clip(
        (float(a[6]) - float(ACTION_MIN[6])) /
        (float(ACTION_MAX[6]) - float(ACTION_MIN[6]) + 1e-8) * 100,
        0, 100
    ))
    
    return m750, grip_pct


# ══════════════════════════════════════════════════════════════
# Main Inference Loop
# ══════════════════════════════════════════════════════════════

try:
    from pymycobot import MyArm
    arm = MyArm(PORT, 115200)
    time.sleep(2)
    print(f"✅ M750 connected: {PORT}")

    # Initialize robot
    print("\n🔧 Initializing robot...")
    arm.power_on()
    time.sleep(2)
    arm.set_fresh_mode(1)
    time.sleep(0.5)
    
    print("⚠️  Note: M750 get_coords() returns -1 (known limitation)")
    print("    Commands sent 'blind' - watch robot physically!\n")

    # Setup camera or use dataset images
    if USE_CAMERA:
        print("🎥 Initializing camera...")
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("⚠️  Camera failed, using dataset images")
            USE_CAMERA = False
    
    confirm = input("Start RDT2-style inference on M750? (yes/no): ")
    if confirm.lower() != 'yes':
        print("Aborted.")
        exit()

    print(f"\n🚀 Running {N_STEPS} steps...")
    print(f"   Instruction: '{INSTRUCTION}'")
    print(f"   Using RDT2-style prediction with action chunks\n")
    
    total_frames = len(root['data']['camera0_rgb'])
    
    for step in range(N_STEPS):
        print(f"\n{'='*70}")
        print(f"Step {step+1}/{N_STEPS}")
        
        # Get image
        if USE_CAMERA:
            ret, frame = cap.read()
            if ret:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            else:
                # Fallback to dataset
                frame_idx = (500 + step * 100) % total_frames
                img_arr = np.array(root['data']['camera0_rgb'][frame_idx], dtype=np.uint8)
                img = Image.fromarray(img_arr)
        else:
            # Use dataset images
            frame_idx = (500 + step * 100) % total_frames
            img_arr = np.array(root['data']['camera0_rgb'][frame_idx], dtype=np.uint8)
            img = Image.fromarray(img_arr)
            print(f"  Using dataset frame {frame_idx}")
        
        # RDT2-style prediction
        try:
            action_chunk = predict_action_rdt2_style(img, INSTRUCTION)
            # action_chunk shape: (24, 10) - 24 future timesteps
            
            # Use first timestep (like RDT2 does in practice)
            action_t0 = action_chunk[0]  # (10,)
            
            # Convert to M750 format
            coords, grip_val = convert_to_m750_format(action_t0)
            
            print(f"  Predicted action chunk: {action_chunk.shape}")
            print(f"  Using timestep 0: {np.round(action_t0, 3)}")
            print(f"  M750 command: {coords} grip={grip_val}")
            
            # Safety check
            safe = (-400 < coords[0] < 400 and
                    -400 < coords[1] < 400 and
                      50 < coords[2] < 600)
            
            if safe:
                # Send command
                arm.send_coords(coords, speed=SPEED, mode=0)
                time.sleep(SLEEP)
                
                arm.set_gripper_value(grip_val, speed=50)
                time.sleep(0.5)
                
                print(f"  ✅ Command sent")
            else:
                print(f"  ⚠️  UNSAFE - skipped")
                
        except Exception as e:
            print(f"  ❌ Prediction error: {e}")
            import traceback
            traceback.print_exc()

    print("\n" + "="*70)
    print("✅ Inference complete!")
    
    if USE_CAMERA:
        cap.release()
    
    # Return home
    print("\n🏠 Returning to home...")
    arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
    time.sleep(3)

except KeyboardInterrupt:
    print("\n⚠️  Interrupted")
    try:
        if USE_CAMERA:
            cap.release()
        arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
        time.sleep(3)
    except:
        pass

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

✅ M750 connected: /dev/ttyACM1

🔧 Initializing robot...
⚠️  Note: M750 get_coords() returns -1 (known limitation)
    Commands sent 'blind' - watch robot physically!


🚀 Running 20 steps...
   Instruction: 'PICK THE MARKER PLACE IN THE BOX'
   Using RDT2-style prediction with action chunks


Step 1/20
  Using dataset frame 500
  Predicted action chunk: (24, 10)
  Using timestep 0: [ 2.40000e-02 -2.21000e-01 -7.90000e-02  5.17094e+02  1.19570e+01
 -5.85300e+00 -3.04330e+01  5.72760e+02  1.93170e+01 -4.08000e-01]
  M750 command: [11.5, -136.9, 119.3, 180.0, 179.9, -48.6] grip=0
  ✅ Command sent

Step 2/20
  Using dataset frame 600
  Predicted action chunk: (24, 10)
  Using timestep 0: [ 1.40000e-02 -2.03000e-01 -9.50000e-02  4.37078e+02  4.68600e+00
  1.87940e+01 -2.29190e+01  4.88699e+02  9.74300e+00  7.75000e-01]
  M750 command: [10.0, -134.3, 117.7, 180.0, 179.9, 67.4] grip=0
  ✅ Command sent

Step 3/20
  Using dataset frame 700
  Predicted action chunk: (24, 10)
  Using timestep 0: [

In [12]:
# CELL 6 — FIXED with corrected safety check and debugging

import time, numpy as np, torch
from PIL import Image

PORT = '/dev/ttyACM1'
SPEED = 30
N_STEPS = 20
SLEEP = 2.5

def predict_action_rdt2_style(image, instruction):
    """RDT2-style inference"""
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    image = image.resize((384, 384))
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": instruction}
        ]
    }]
    
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[text], images=[image], return_tensors="pt"
    ).to("cuda", torch.bfloat16)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    
    input_len = inputs["input_ids"].shape[1]
    generated = output_ids[:, input_len:].cpu()
    token_ids = generated[0].numpy()
    
    del inputs
    torch.cuda.empty_cache()
    
    SPECIAL = {151643, 151644, 151645, 151646, 151650, 151651,
               151652, 151653, 151654, 151655, 151656}
    clean = [t for t in token_ids.tolist() if t not in SPECIAL]
    
    BASE = 151000
    pos_len = vae.pos_id_len
    rot_len = vae.rot_id_len
    grip_len = getattr(vae, 'grip_id_len', 3)
    total = pos_len + rot_len + grip_len
    
    remapped = [max(0, t - BASE) for t in clean[:total]]
    ids_t = torch.tensor(remapped, dtype=torch.long).unsqueeze(0)
    
    vae_cpu = vae.cpu()
    with torch.no_grad():
        result = vae_cpu.decode(ids_t)
    
    action_chunk = result[0].float().numpy()
    return action_chunk


def convert_to_m750_format(action_chunk_step):
    """Convert to M750 format"""
    raw_action = action_chunk_step[:7]
    
    # Unnormalize
    a = raw_action * ACTION_STD[:7] + ACTION_MEAN[:7]
    a = np.clip(a, ACTION_MIN[:7], ACTION_MAX[:7])
    
    # Convert
    m750 = [
        round(float(a[0]) * 1000, 1),
        round(float(a[1]) * 1000, 1),
        round(float(a[2]) * 1000, 1),
        round(float(np.degrees(a[3])), 1),
        round(float(np.degrees(a[4])), 1),
        round(float(np.degrees(a[5])), 1),
    ]
    
    grip_pct = int(np.clip(
        (float(a[6]) - float(ACTION_MIN[6])) /
        (float(ACTION_MAX[6]) - float(ACTION_MIN[6]) + 1e-8) * 100,
        0, 100
    ))
    
    return m750, grip_pct, a


try:
    from pymycobot import MyArm
    arm = MyArm(PORT, 115200)
    time.sleep(2)
    print(f"✅ M750 connected: {PORT}")

    arm.power_on()
    time.sleep(2)
    arm.set_fresh_mode(1)
    time.sleep(0.5)
    
    print("\n⚠️  Robot ready - WATCH PHYSICALLY during movements!\n")

    confirm = input("Start inference? (yes/no): ")
    if confirm.lower() != 'yes':
        exit()

    print(f"\n🚀 Running {N_STEPS} steps...")
    print(f"   Instruction: '{INSTRUCTION}'\n")
    
    total_frames = len(root['data']['camera0_rgb'])
    
    for step in range(N_STEPS):
        print(f"\n{'='*70}")
        print(f"Step {step+1}/{N_STEPS}")
        
        frame_idx = (500 + step * 100) % total_frames
        img_arr = np.array(root['data']['camera0_rgb'][frame_idx], dtype=np.uint8)
        img = Image.fromarray(img_arr)
        print(f"  Frame {frame_idx}")
        
        try:
            action_chunk = predict_action_rdt2_style(img, INSTRUCTION)
            action_t0 = action_chunk[0]
            
            coords, grip_val, unnorm_action = convert_to_m750_format(action_t0)
            
            print(f"  Unnorm: pos=[{unnorm_action[0]:.3f}, {unnorm_action[1]:.3f}, {unnorm_action[2]:.3f}]m " +
                  f"rot=[{np.degrees(unnorm_action[3]):.1f}, {np.degrees(unnorm_action[4]):.1f}, {np.degrees(unnorm_action[5]):.1f}]° " +
                  f"grip={unnorm_action[6]:.4f}m")
            print(f"  M750: {coords} grip={grip_val}%")
            
            # Safety check with detailed output
            checks = {
                'x': -400 < coords[0] < 400,
                'y': -400 < coords[1] < 400,
                'z': 50 < coords[2] < 600,
                'rx': -180 <= coords[3] <= 180,
                'ry': -180 <= coords[4] <= 180,
                'rz': -180 <= coords[5] <= 180,
            }
            
            safe = all(checks.values())
            
            if not safe:
                print(f"  ⚠️  UNSAFE! Failed checks:")
                for key, passed in checks.items():
                    if not passed:
                        idx = ['x','y','z','rx','ry','rz'].index(key)
                        print(f"     {key}: {coords[idx]} (range depends on axis)")
            else:
                print(f"  ✅ Safety check PASSED")
                
                # Send command
                arm.send_coords(coords, speed=SPEED, mode=0)
                print(f"  ⏳ Waiting {SLEEP}s for movement...")
                time.sleep(SLEEP)
                
                arm.set_gripper_value(grip_val, speed=50)
                time.sleep(0.5)
                
                print(f"  ✅ Command executed")
                
        except Exception as e:
            print(f"  ❌ Error: {e}")
            import traceback
            traceback.print_exc()

    print("\n" + "="*70)
    print("✅ Complete!")
    
    print("\n🏠 Returning home...")
    arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
    time.sleep(3)

except KeyboardInterrupt:
    print("\n⚠️  Interrupted")
    try:
        arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
        time.sleep(3)
    except:
        pass

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

✅ M750 connected: /dev/ttyACM1

⚠️  Robot ready - WATCH PHYSICALLY during movements!


🚀 Running 20 steps...
   Instruction: 'PICK THE MARKER PLACE IN THE BOX'


Step 1/20
  Frame 500
  Unnorm: pos=[0.012, -0.137, 0.119]m rot=[180.0, 179.9, -48.6]° grip=0.0065m
  M750: [11.5, -136.9, 119.3, 180.0, 179.9, -48.6] grip=0%
  ✅ Safety check PASSED
  ⏳ Waiting 2.5s for movement...
  ✅ Command executed

Step 2/20
  Frame 600
  Unnorm: pos=[0.010, -0.134, 0.118]m rot=[180.0, 179.9, 67.4]° grip=0.0065m
  M750: [10.0, -134.3, 117.7, 180.0, 179.9, 67.4] grip=0%
  ✅ Safety check PASSED
  ⏳ Waiting 2.5s for movement...
  ✅ Command executed

Step 3/20
  Frame 700
  Unnorm: pos=[0.012, -0.134, 0.118]m rot=[180.0, 179.9, 67.4]° grip=0.0065m
  M750: [11.5, -134.1, 118.1, 180.0, 179.9, 67.4] grip=0%
  ✅ Safety check PASSED
  ⏳ Waiting 2.5s for movement...
  ✅ Command executed

Step 4/20
  Frame 800
  Unnorm: pos=[0.010, -0.137, 0.117]m rot=[180.0, 179.9, -48.6]° grip=0.0065m
  M750: [9.7, -136.8, 117.1,

In [13]:
# Quick test - use ground truth from diverse frames
test_frames = [0, 10000, 20000, 30000, 40000, 50000, 60000, 70000]

for i, frame_idx in enumerate(test_frames):
    true_action = actions[frame_idx]
    
    m750 = [
        round(float(true_action[0])*1000, 1),
        round(float(true_action[1])*1000, 1),
        round(float(true_action[2])*1000, 1),
        round(float(np.degrees(true_action[3])), 1),
        round(float(np.degrees(true_action[4])), 1),
        round(float(np.degrees(true_action[5])), 1),
    ]
    
    grip = int(np.clip((float(true_action[6]) - float(ACTION_MIN[6])) /
                       (float(ACTION_MAX[6]) - float(ACTION_MIN[6]) + 1e-8) * 100, 0, 100))
    
    print(f"\nFrame {frame_idx}: {m750} grip={grip}")
    
    safe = (-400 < m750[0] < 400 and -400 < m750[1] < 400 and 50 < m750[2] < 600)
    if safe:
        input(f"Send this command? (Enter to continue)")
        arm.send_coords(m750, speed=30, mode=0)
        time.sleep(3)
        


Frame 0: [73.4, -238.3, 256.7, -135.5, 0.4, -5.3] grip=97

Frame 10000: [-21.1, 80.2, -0.5, -141.1, -90.7, 2.9] grip=10

Frame 20000: [259.0, -111.9, 1.9, -174.3, -2.4, 4.2] grip=85

Frame 30000: [-13.5, -267.3, 318.5, -136.9, 1.8, 5.3] grip=73

Frame 40000: [-179.2, -143.9, 176.9, -141.7, -26.1, 31.4] grip=9

Frame 50000: [107.5, 23.2, 71.3, -156.1, -3.3, 5.9] grip=82

Frame 60000: [198.4, 163.6, 8.2, -175.2, -7.7, 3.9] grip=78

Frame 70000: [-209.8, -234.2, 68.7, -126.7, 107.2, -2.7] grip=79


USE DIRECTLY THE PRETRAINERD MODEL


In [2]:
import os, sys, numpy as np
from PIL import Image

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
NORMALIZER   = "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
INSTRUCTION  = "Pick up the marker and place it in the box."  # RDT2 format

sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))

os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
os.environ['PYTORCH_CUDA_ALLOC_CONF']          = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("✅ Setup done")
print(f"   Using base RDT2-VQ model (no fine-tuning)")
print(f"   Instruction: '{INSTRUCTION}'")

✅ Setup done
   Using base RDT2-VQ model (no fine-tuning)
   Instruction: 'Pick up the marker and place it in the box.'


In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

# Use the EXACT same imports that worked in your original code
import sys
import os

RDT2_DIR = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))

# Import using the paths that already worked for you
from models.normalizer.normalizer import LinearNormalizer
from models.multivqvae import MultiVQVAE

print("[1/4] Loading base RDT2-VQ model (zero-shot)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "robotics-diffusion-transformer/RDT2-VQ",  # Base model, NOT fine-tuned
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    device_map="cuda"
).eval()
print("      ✅ Base model loaded (zero-shot deployment)")

print("[2/4] Loading processor...")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True
)

print("[3/4] Loading VAE...")
vae = MultiVQVAE.from_pretrained(
    "robotics-diffusion-transformer/RVQActionTokenizer"
).cuda().eval()

print("\n✅ ALL MODELS LOADED (using base RDT2-VQ)!")

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[1/4] Loading base RDT2-VQ model (zero-shot)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.51it/s]


      ✅ Base model loaded (zero-shot deployment)
[2/4] Loading processor...
[3/4] Loading VAE...

✅ ALL MODELS LOADED (using base RDT2-VQ)!


In [3]:
import zarr, numpy as np
from PIL import Image

root = zarr.open(DATASET_PATH, mode='r')

# Load action stats
pos  = np.array(root['data']['robot0_eef_pos'])
rot  = np.array(root['data']['robot0_eef_rot_axis_angle'])
grip = np.array(root['data']['robot0_gripper_width'])
actions = np.concatenate([pos, rot, grip], axis=-1)

ACTION_MEAN = actions.mean(axis=0)
ACTION_STD  = actions.std(axis=0)
ACTION_STD  = np.where(ACTION_STD < 1e-8, 1.0, ACTION_STD)
ACTION_MIN  = actions.min(axis=0)
ACTION_MAX  = actions.max(axis=0)

print(f"✅ Dataset stats loaded")
print(f"   mean: {np.round(ACTION_MEAN, 4)}")
print(f"   std:  {np.round(ACTION_STD,  4)}")

# Load test image
img_arr = np.array(root['data']['camera0_rgb'][500], dtype=np.uint8)
img     = Image.fromarray(img_arr).resize((768, 384))
print(f"✅ Image loaded: {img_arr.shape} → 768x384")

✅ Dataset stats loaded
   mean: [ 0.0077 -0.1062  0.1275 -2.1636 -0.1622  0.0445  0.0265]
   std:  [0.157  0.1385 0.104  1.3838 0.8236 0.1868 0.0119]
✅ Image loaded: (224, 224, 3) → 768x384


In [4]:
import numpy as np, torch

# Special tokens
SPECIAL = {151643, 151644, 151645, 151646, 151650, 151651,
           151652, 151653, 151654, 151655, 151656}

# VAE token parameters
BASE     = 151000
pos_len  = vae.pos_id_len     # 18
rot_len  = vae.rot_id_len     # 6
grip_len = getattr(vae, 'grip_id_len', 3)  # 3
total    = pos_len + rot_len + grip_len     # 27

print(f"✅ Decoding setup complete")
print(f"   VAE expects {total} tokens total")

✅ Decoding setup complete
   VAE expects 27 tokens total


In [ ]:
import time, numpy as np, torch
from PIL import Image
import cv2

PORT = '/dev/ttyACM1'
SPEED = 30
N_STEPS = 30  # More steps to see pattern
SLEEP = 2.5
USE_CAMERA = False  # Set True when camera works

def predict_action_base_model(image, instruction):
    """Predict using base RDT2-VQ (zero-shot)"""
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    image = image.resize((384, 384))
    
    # Format message
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": instruction}
        ]
    }]
    
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[text], 
        images=[image], 
        return_tensors="pt"
    ).to("cuda", torch.bfloat16)
    
    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )
    
    input_len = inputs["input_ids"].shape[1]
    generated = output_ids[:, input_len:].cpu()
    token_ids = generated[0].numpy()
    
    del inputs
    torch.cuda.empty_cache()
    
    # Decode tokens
    clean = [t for t in token_ids.tolist() if t not in SPECIAL]
    remapped = [max(0, t - BASE) for t in clean[:total]]
    ids_t = torch.tensor(remapped, dtype=torch.long).unsqueeze(0)
    
    vae_cpu = vae.cpu()
    with torch.no_grad():
        result = vae_cpu.decode(ids_t)  # (1, 24, 20)
    
    action_chunk = result[0].float().numpy()  # (24, 20)
    return action_chunk


def convert_to_m750(action_step):
    """Convert action to M750 format"""
    raw = action_step[:7]
    
    # Unnormalize
    a = raw * ACTION_STD[:7] + ACTION_MEAN[:7]
    a = np.clip(a, ACTION_MIN[:7], ACTION_MAX[:7])
    
    # Convert to M750
    m750 = [
        round(float(a[0]) * 1000, 1),  # m to mm
        round(float(a[1]) * 1000, 1),
        round(float(a[2]) * 1000, 1),
        round(float(np.degrees(a[3])), 1),  # rad to deg
        round(float(np.degrees(a[4])), 1),
        round(float(np.degrees(a[5])), 1),
    ]
    
    # Gripper
    grip_pct = int(np.clip(
        (float(a[6]) - float(ACTION_MIN[6])) /
        (float(ACTION_MAX[6]) - float(ACTION_MIN[6]) + 1e-8) * 100,
        0, 100
    ))
    
    return m750, grip_pct, a


try:
    from pymycobot import MyArm
    arm = MyArm(PORT, 115200)
    time.sleep(2)
    print(f"✅ M750 connected: {PORT}")

    # Initialize
    arm.power_on()
    time.sleep(2)
    arm.set_fresh_mode(1)
    time.sleep(0.5)
    
    print("\n" + "="*70)
    print("🤖 ZERO-SHOT DEPLOYMENT TEST")
    print("="*70)
    print("Using BASE RDT2-VQ model (no fine-tuning)")
    print("This tests the model's zero-shot generalization capability")
    print("="*70)
    print(f"\n⚠️  Commands sent blind - WATCH ROBOT PHYSICALLY!\n")

    confirm = input(f"Start zero-shot inference for: '{INSTRUCTION}'? (yes/no): ")
    if confirm.lower() != 'yes':
        exit()

    print(f"\n🚀 Running {N_STEPS} steps...\n")
    
    total_frames = len(root['data']['camera0_rgb'])
    
    # Track predictions to see if they vary
    all_positions = []
    
    for step in range(N_STEPS):
        print(f"{'='*70}")
        print(f"Step {step+1}/{N_STEPS}")
        
        # Get image
        if USE_CAMERA:
            # Add camera code here when ready
            pass
        else:
            # Use diverse frames from dataset
            frame_idx = (500 + step * 200) % total_frames
            img_arr = np.array(root['data']['camera0_rgb'][frame_idx], dtype=np.uint8)
            img = Image.fromarray(img_arr)
            print(f"  Frame {frame_idx}")
        
        try:
            # Predict with base model
            action_chunk = predict_action_base_model(img, INSTRUCTION)
            action_t0 = action_chunk[0]  # Use first timestep
            
            # Convert
            coords, grip_val, unnorm = convert_to_m750(action_t0)
            
            all_positions.append(coords[:3])  # Track xyz
            
            print(f"  Unnorm: pos=[{unnorm[0]:.3f}, {unnorm[1]:.3f}, {unnorm[2]:.3f}]m")
            print(f"          rot=[{np.degrees(unnorm[3]):.1f}, {np.degrees(unnorm[4]):.1f}, {np.degrees(unnorm[5]):.1f}]°")
            print(f"          grip={unnorm[6]:.4f}m ({grip_val}%)")
            print(f"  M750: {coords} grip={grip_val}%")
            
            # Safety check
            safe = (-400 < coords[0] < 400 and
                    -400 < coords[1] < 400 and
                      50 < coords[2] < 600 and
                    -180 <= coords[3] <= 180 and
                    -180 <= coords[4] <= 180 and
                    -180 <= coords[5] <= 180)
            
            if safe:
                print(f"  ✅ Safe - sending command")
                
                arm.send_coords(coords, speed=SPEED, mode=0)
                time.sleep(SLEEP)
                
                arm.set_gripper_value(grip_val, speed=50)
                time.sleep(0.5)
                
                print(f"  ✅ Executed")
            else:
                print(f"  ⚠️  UNSAFE - skipped")
                
        except Exception as e:
            print(f"  ❌ Error: {e}")
            import traceback
            traceback.print_exc()
    
    # Analyze prediction diversity
    print("\n" + "="*70)
    print("📊 PREDICTION ANALYSIS")
    print("="*70)
    
    if len(all_positions) > 0:
        all_pos = np.array(all_positions)
        print(f"Position ranges:")
        print(f"  X: [{all_pos[:,0].min():.1f}, {all_pos[:,0].max():.1f}] mm (range: {all_pos[:,0].max()-all_pos[:,0].min():.1f}mm)")
        print(f"  Y: [{all_pos[:,1].min():.1f}, {all_pos[:,1].max():.1f}] mm (range: {all_pos[:,1].max()-all_pos[:,1].min():.1f}mm)")
        print(f"  Z: [{all_pos[:,2].min():.1f}, {all_pos[:,2].max():.1f}] mm (range: {all_pos[:,2].max()-all_pos[:,2].min():.1f}mm)")
        
        total_range = (all_pos.max(axis=0) - all_pos.min(axis=0)).sum()
        print(f"\nTotal position diversity: {total_range:.1f}mm")
        
        if total_range < 50:
            print("⚠️  WARNING: Very low diversity - model may need better training")
        elif total_range < 200:
            print("⚠️  Moderate diversity - model is being conservative")
        else:
            print("✅ Good diversity - model is making varied predictions")
    
    print("="*70)
    print("✅ Inference complete!")
    
    # Return home
    print("\n🏠 Returning to home...")
    arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
    time.sleep(3)

except KeyboardInterrupt:
    print("\n⚠️  Interrupted")
    try:
        arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
        time.sleep(3)
    except:
        pass

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
    try:
        arm.send_coords([150, 0, 300, 0, 0, 0], speed=20, mode=0)
        time.sleep(3)
    except:
        pass

✅ M750 connected: /dev/ttyACM1

🤖 ZERO-SHOT DEPLOYMENT TEST
Using BASE RDT2-VQ model (no fine-tuning)
This tests the model's zero-shot generalization capability

⚠️  Commands sent blind - WATCH ROBOT PHYSICALLY!


🚀 Running 30 steps...

Step 1/30
  Frame 500


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


  Unnorm: pos=[0.007, -0.137, 0.116]m
          rot=[180.0, 109.0, 22.1]°
          grip=0.0065m (0%)
  M750: [6.7, -137.0, 116.2, 180.0, 109.0, 22.1] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 2/30
  Frame 700
  Unnorm: pos=[0.008, -0.139, 0.120]m
          rot=[180.0, 149.6, -28.3]°
          grip=0.0065m (0%)
  M750: [8.3, -138.7, 119.6, 180.0, 149.6, -28.3] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 3/30
  Frame 900
  Unnorm: pos=[0.011, -0.136, 0.118]m
          rot=[180.0, -179.9, -32.9]°
          grip=0.0065m (0%)
  M750: [11.4, -135.8, 117.6, 180.0, -179.9, -32.9] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 4/30
  Frame 1100
  Unnorm: pos=[0.012, -0.134, 0.118]m
          rot=[180.0, 179.9, -13.9]°
          grip=0.0065m (0%)
  M750: [11.9, -133.8, 117.8, 180.0, 179.9, -13.9] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 5/30
  Frame 1300
  Unnorm: pos=[0.011, -0.132, 0.117]m
          rot=[180.0, 179.9, 67.4]°
          grip=0.0065m (0%)
  

Traceback (most recent call last):
  File "/tmp/ipykernel_93385/3712563989.py", line 140, in <module>
    action_chunk = predict_action_base_model(img, INSTRUCTION)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_93385/3712563989.py", line 58, in predict_action_base_model
    result = vae_cpu.decode(ids_t)  # (1, 24, 20)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/home/rishabh/Downloads/umi-pipeline-training/RDT2/vqvae/models/multivqvae.py", line 146, in decode
    x = apply_act_dim(x, x_grip_recon, "grip")
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/rishabh/Downloads/umi-pipeline-training/RDT2/vqvae/models/multivqvae.py", line 186, in apply_act_dim
    x[..., 9:10] = y[..., :1]
    ~^^^^^^^^^^^
RuntimeError: The expanded size of the tensor (24) must match the existing size (16) at non-singleton dimension 1.  Target sizes: [1, 24, 1].  Tensor sizes: [16, 1]


  Unnorm: pos=[0.013, -0.137, 0.118]m
          rot=[180.0, 37.0, -48.6]°
          grip=0.0065m (0%)
  M750: [13.4, -136.7, 117.6, 180.0, 37.0, -48.6] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 19/30
  Frame 4100
  Unnorm: pos=[0.008, -0.135, 0.118]m
          rot=[180.0, -123.0, 40.6]°
          grip=0.0065m (0%)
  M750: [8.5, -134.8, 117.5, 180.0, -123.0, 40.6] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 20/30
  Frame 4300
  Unnorm: pos=[0.010, -0.135, 0.117]m
          rot=[180.0, 179.9, 7.7]°
          grip=0.0065m (0%)
  M750: [10.0, -135.3, 117.2, 180.0, 179.9, 7.7] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 21/30
  Frame 4500
  Unnorm: pos=[0.010, -0.132, 0.117]m
          rot=[180.0, -129.7, -48.6]°
          grip=0.0065m (0%)
  M750: [10.2, -132.3, 116.9, 180.0, -129.7, -48.6] grip=0%
  ✅ Safe - sending command
  ✅ Executed
Step 22/30
  Frame 4700
  Unnorm: pos=[0.009, -0.137, 0.116]m
          rot=[180.0, 41.8, 67.4]°
          grip=0.0065m (0%)


: 

In [2]:
# ═══════════════════════════════════════════════════════════════
# FULL DIAGNOSTIC SCRIPT - Find All Problems
# ═══════════════════════════════════════════════════════════════

import os, sys, torch, numpy as np
import zarr
from PIL import Image

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
OUTPUT_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/outputs/vqvla-sft-m750-pick-place"
NORM_PATH    = "/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"

sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))
sys.path.insert(0, os.path.join(RDT2_DIR, 'models'))
os.chdir(RDT2_DIR)

print("=" * 70)
print("🔍 RDT2 FULL DIAGNOSTIC REPORT")
print("=" * 70)

# ──────────────────────────────────────────────────────────────
# 1. GPU STATUS
# ──────────────────────────────────────────────────────────────
print("\n📊 [1/7] GPU STATUS")
print("-" * 40)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gpu_name    = torch.cuda.get_device_name(0)
    total_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_mem    = torch.cuda.mem_get_info(0)[0] / 1e9
    used_mem    = total_mem - free_mem
    
    print(f"  GPU:          {gpu_name}")
    print(f"  Total VRAM:   {total_mem:.1f} GB")
    print(f"  Used VRAM:    {used_mem:.1f} GB")
    print(f"  Free VRAM:    {free_mem:.1f} GB")
    
    if free_mem < 5:
        print(f"  ❌ CRITICAL: Only {free_mem:.1f}GB free - will OOM!")
        print(f"     FIX: Restart kernel to free GPU memory")
    elif free_mem < 15:
        print(f"  ⚠️  WARNING: Only {free_mem:.1f}GB free - might OOM")
        print(f"     FIX: Restart kernel before training")
    else:
        print(f"  ✅ GPU memory looks good")
else:
    print("  ❌ No CUDA GPU found!")

# ──────────────────────────────────────────────────────────────
# 2. DATASET INSPECTION
# ──────────────────────────────────────────────────────────────
print("\n📊 [2/7] DATASET INSPECTION")
print("-" * 40)

try:
    root = zarr.open(DATASET_PATH, mode='r')
    
    print(f"  Dataset path: {DATASET_PATH}")
    print(f"  Keys: {list(root['data'].keys())}")
    
    n_frames = len(root['data']['camera0_rgb'])
    print(f"  Total frames: {n_frames:,}")
    
    # Check image shape
    img = root['data']['camera0_rgb'][0]
    print(f"  Image shape:  {img.shape}")
    print(f"  Image dtype:  {img.dtype}")
    
    # Check action data
    pos  = np.array(root['data']['robot0_eef_pos'])
    rot  = np.array(root['data']['robot0_eef_rot_axis_angle'])
    grip = np.array(root['data']['robot0_gripper_width'])
    
    print(f"\n  Action Statistics:")
    print(f"  Position (x,y,z):")
    print(f"    Range X: [{pos[:,0].min():.3f}, {pos[:,0].max():.3f}] m")
    print(f"    Range Y: [{pos[:,1].min():.3f}, {pos[:,1].max():.3f}] m")
    print(f"    Range Z: [{pos[:,2].min():.3f}, {pos[:,2].max():.3f}] m")
    print(f"  Rotation (axis-angle):")
    print(f"    Range RX: [{np.degrees(rot[:,0].min()):.1f}, {np.degrees(rot[:,0].max()):.1f}] deg")
    print(f"    Range RY: [{np.degrees(rot[:,1].min()):.1f}, {np.degrees(rot[:,1].max()):.1f}] deg")
    print(f"    Range RZ: [{np.degrees(rot[:,2].min()):.1f}, {np.degrees(rot[:,2].max()):.1f}] deg")
    print(f"  Gripper:")
    print(f"    Range: [{grip.min():.4f}, {grip.max():.4f}] m")
    print(f"    Min width: {grip.min()*1000:.1f}mm")
    print(f"    Max width: {grip.max()*1000:.1f}mm")
    
    # Check for NaN/Inf in dataset
    actions = np.concatenate([pos, rot, grip], axis=-1)
    nan_count = np.isnan(actions).sum()
    inf_count = np.isinf(actions).sum()
    
    if nan_count > 0:
        print(f"  ❌ NaN values in actions: {nan_count}")
    elif inf_count > 0:
        print(f"  ❌ Inf values in actions: {inf_count}")
    else:
        print(f"  ✅ No NaN/Inf in dataset")
    
    # Check dataset diversity
    pos_range = pos.max(axis=0) - pos.min(axis=0)
    print(f"\n  Dataset diversity:")
    print(f"    Position range: [{pos_range[0]*1000:.1f}, {pos_range[1]*1000:.1f}, {pos_range[2]*1000:.1f}] mm")
    if pos_range.sum() < 0.1:
        print(f"    ❌ CRITICAL: Very low diversity - robot barely moved during data collection!")
    else:
        print(f"    ✅ Good diversity in training data")

except Exception as e:
    print(f"  ❌ Dataset error: {e}")

# ──────────────────────────────────────────────────────────────
# 3. NORMALIZER INSPECTION
# ──────────────────────────────────────────────────────────────
print("\n📊 [3/7] NORMALIZER INSPECTION")
print("-" * 40)

try:
    from models.normalizer.normalizer import LinearNormalizer
    normalizer = LinearNormalizer.load(NORM_PATH)
    
    print(f"  ✅ Normalizer loaded successfully")
    print(f"  Type: {type(normalizer)}")
    
    if hasattr(normalizer, '__getitem__'):
        action_norm = normalizer['action']
        print(f"\n  Action normalizer type: {type(action_norm)}")
        
        if hasattr(action_norm, 'mean'):
            mean = action_norm.mean
            std  = action_norm.std
            print(f"  Mean: {np.round(mean, 4) if hasattr(mean, '__len__') else mean}")
            print(f"  Std:  {np.round(std, 4) if hasattr(std, '__len__') else std}")
            
            if hasattr(action_norm, 'input_min'):
                print(f"  Min:  {np.round(action_norm.input_min, 4)}")
                print(f"  Max:  {np.round(action_norm.input_max, 4)}")
        
        # Compare normalizer with dataset
        if 'actions' in dir():
            dataset_mean = actions.mean(axis=0)
            dataset_std  = actions.std(axis=0)
            print(f"\n  Normalizer vs Dataset comparison:")
            print(f"  If these differ greatly → prediction issues!")
            
except Exception as e:
    print(f"  ❌ Normalizer error: {e}")
    import traceback
    traceback.print_exc()

# ──────────────────────────────────────────────────────────────
# 4. MODEL CHECKPOINT INSPECTION
# ──────────────────────────────────────────────────────────────
print("\n📊 [4/7] CHECKPOINT INSPECTION")
print("-" * 40)

try:
    checkpoints = sorted([
        d for d in os.listdir(OUTPUT_DIR)
        if d.startswith('checkpoint-')
    ])
    
    print(f"  Available checkpoints:")
    for ckpt in checkpoints:
        ckpt_path = os.path.join(OUTPUT_DIR, ckpt)
        
        # Get size
        total_size = sum(
            os.path.getsize(os.path.join(dirpath, f))
            for dirpath, dirnames, filenames in os.walk(ckpt_path)
            for f in filenames
        ) / 1e6  # MB
        
        # Get files
        files = os.listdir(ckpt_path)
        
        print(f"\n  📁 {ckpt}:")
        print(f"     Size: {total_size:.1f} MB")
        print(f"     Files: {files}")
        
        # Check adapter config
        adapter_config = os.path.join(ckpt_path, 'adapter_config.json')
        if os.path.exists(adapter_config):
            import json
            with open(adapter_config) as f:
                config = json.load(f)
            print(f"     LoRA rank: {config.get('r', 'unknown')}")
            print(f"     LoRA alpha: {config.get('lora_alpha', 'unknown')}")
            print(f"     ✅ Valid PEFT checkpoint")
        else:
            print(f"     ⚠️  No adapter_config.json found")

except Exception as e:
    print(f"  ❌ Checkpoint error: {e}")

# ──────────────────────────────────────────────────────────────
# 5. TRAINING LOG INSPECTION
# ──────────────────────────────────────────────────────────────
print("\n📊 [5/7] TRAINING LOG INSPECTION")
print("-" * 40)

log_files = [
    "/home/rishabh/Downloads/umi-pipeline-training/train_m750-pick-place.log",
    "/home/rishabh/Downloads/umi-pipeline-training/train_resume.log",
    "/home/rishabh/Downloads/umi-pipeline-training/train.log",
]

for log_path in log_files:
    if os.path.exists(log_path):
        print(f"\n  Log: {log_path}")
        
        with open(log_path) as f:
            lines = f.readlines()
        
        print(f"  Total lines: {len(lines)}")
        
        # Extract loss values
        loss_lines = [l for l in lines if "'loss'" in l or '"loss"' in l]
        if loss_lines:
            print(f"\n  Training progress (first 5 and last 5 loss values):")
            
            for line in loss_lines[:5]:
                print(f"    {line.strip()}")
            
            if len(loss_lines) > 5:
                print(f"    ...")
                for line in loss_lines[-5:]:
                    print(f"    {line.strip()}")
            
            # Check if loss went to zero too fast
            first_loss_line = loss_lines[0]
            last_loss_line  = loss_lines[-1]
            print(f"\n  ⚠️  Check: Did loss go to 0 too quickly?")
            print(f"    First loss: {first_loss_line.strip()}")
            print(f"    Last loss:  {last_loss_line.strip()}")

# ──────────────────────────────────────────────────────────────
# 6. VAE RECONSTRUCTION TEST
# ──────────────────────────────────────────────────────────────
print("\n📊 [6/7] VAE RECONSTRUCTION TEST")
print("-" * 40)

try:
    from models.multivqvae import MultiVQVAE
    
    vae = MultiVQVAE.from_pretrained(
        "robotics-diffusion-transformer/RVQActionTokenizer"
    ).eval()
    
    print(f"  ✅ VAE loaded")
    print(f"  pos_id_len:  {vae.pos_id_len}")
    print(f"  rot_id_len:  {vae.rot_id_len}")
    print(f"  grip_id_len: {vae.grip_id_len}")
    
    # Test reconstruction with real dataset action
    root = zarr.open(DATASET_PATH, mode='r')
    pos  = np.array(root['data']['robot0_eef_pos'][:24])
    rot  = np.array(root['data']['robot0_eef_rot_axis_angle'][:24])
    grip = np.array(root['data']['robot0_gripper_width'][:24])
    
    # Create action chunk
    action_chunk = np.concatenate([pos, rot, grip], axis=-1)
    print(f"\n  Test action chunk shape: {action_chunk.shape}")
    print(f"  First action: pos={np.round(pos[0], 4)}")
    print(f"  First action: grip={grip[0]:.4f}")
    
    # Check if actions are within VAE bounds
    print(f"\n  VAE bounds check:")
    print(f"  Position range in data: [{pos.min():.3f}, {pos.max():.3f}]")
    print(f"  This should be within [-1, 1] for VAE")
    
    if pos.max() > 1.0 or pos.min() < -1.0:
        print(f"  ⚠️  Actions may be outside VAE expected range!")
        print(f"  This could cause poor reconstruction quality")
    else:
        print(f"  ✅ Actions within expected range")

except Exception as e:
    print(f"  ❌ VAE test error: {e}")
    import traceback
    traceback.print_exc()

# ──────────────────────────────────────────────────────────────
# 7. DATASET CONFIG INSPECTION
# ──────────────────────────────────────────────────────────────
print("\n📊 [7/7] DATASET CONFIG INSPECTION")
print("-" * 40)

config_path = os.path.join(RDT2_DIR, "configs/datasets/custom_dataset.yaml")
try:
    import yaml
    with open(config_path) as f:
        config = yaml.safe_load(f)
    
    print(f"  ✅ Config loaded: {config_path}")
    print(f"  Config contents:")
    
    for key, value in config.items():
        print(f"    {key}: {value}")
    
    # Check instruction format
    if 'instruction' in str(config).lower():
        print(f"\n  ✅ Instruction found in config")
    else:
        print(f"\n  ⚠️  No instruction found - check your dataset config!")
        print(f"     This might cause model to not learn task properly")

except Exception as e:
    print(f"  ❌ Config error: {e}")

# ──────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("🎯 DIAGNOSTIC SUMMARY")
print("="*70)

print("""
Most likely causes of poor model performance:

1. OOM Error During Training:
   → Use batch_size=16 + grad_accum=4 (effective=64)
   → Restart kernel before training to free VRAM

2. Loss Went to Zero Too Fast:
   → Model memorized data without learning task
   → Need higher LR + more diverse data

3. Normalizer Mismatch:
   → Official normalizer may not match your dataset
   → Check normalizer stats vs dataset stats

4. VAE Bounds Issue:
   → If actions outside [-1, 1], reconstruction fails
   → Need to check RVQ bounds match your data

5. Instruction Format:
   → Training instruction must match inference instruction
   → Check custom_dataset.yaml for instruction text
""")

print("="*70)
print("Run this script and share the output with your mentor!")
print("="*70)

🔍 RDT2 FULL DIAGNOSTIC REPORT

📊 [1/7] GPU STATUS
----------------------------------------
  GPU:          NVIDIA GeForce RTX 4090
  Total VRAM:   25.3 GB
  Used VRAM:    0.4 GB
  Free VRAM:    24.8 GB
  ✅ GPU memory looks good

📊 [2/7] DATASET INSPECTION
----------------------------------------
  Dataset path: /home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr
  Keys: ['camera0_rgb', 'robot0_demo_end_pose', 'robot0_demo_start_pose', 'robot0_eef_pos', 'robot0_eef_rot_axis_angle', 'robot0_gripper_width']
  Total frames: 78,018
  Image shape:  (224, 224, 3)
  Image dtype:  uint8

  Action Statistics:
  Position (x,y,z):
    Range X: [-0.766, 0.386] m
    Range Y: [-0.874, 0.291] m
    Range Z: [-0.039, 0.378] m
  Rotation (axis-angle):
    Range RX: [-179.9, 180.0] deg
    Range RY: [-179.9, 179.9] deg
    Range RZ: [-48.6, 67.4] deg
  Gripper:
    Range: [0.0065, 0.0438] m
    Min width: 6.5mm
    Max width: 43.8mm
  ✅ No NaN/Inf in dataset

  Dataset diversity:
    Positi

/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ VAE loaded
  pos_id_len:  18
  rot_id_len:  6
  grip_id_len: 3

  Test action chunk shape: (24, 7)
  First action: pos=[ 0.0734 -0.2383  0.2567]
  ❌ VAE test error: unsupported format string passed to numpy.ndarray.__format__

📊 [7/7] DATASET CONFIG INSPECTION
----------------------------------------
  ✅ Config loaded: /home/rishabh/Downloads/umi-pipeline-training/RDT2/configs/datasets/custom_dataset.yaml
  Config contents:
    kwargs: {'instruction_path': '/home/rishabh/Downloads/umi-pipeline-training/shards/instructions.json', 'normalizer_path': '/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt'}
    name: custom/robot_dataset
    shards_dir: /home/rishabh/Downloads/umi-pipeline-training/shards
    type: single

  ✅ Instruction found in config

🎯 DIAGNOSTIC SUMMARY

Most likely causes of poor model performance:

1. OOM Error During Training:
   → Use batch_size=16 + grad_accum=4 (effective=64)
   → Restart kernel before training to free VRAM

2. Loss Went to Zero Too F

Traceback (most recent call last):
  File "/tmp/ipykernel_5445/1953714862.py", line 268, in <module>
    print(f"  First action: grip={grip[0]:.4f}")
                                 ^^^^^^^^^^^^^
TypeError: unsupported format string passed to numpy.ndarray.__format__


In [4]:
import os, sys, json, torch, numpy as np
import zarr

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"

sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))
sys.path.insert(0, os.path.join(RDT2_DIR, 'models'))
os.chdir(RDT2_DIR)

print("=" * 70)
print("🔧 FIXING ALL ISSUES")
print("=" * 70)

# ──────────────────────────────────────────────────────────────
# FIX 1: CHECK BOTH NORMALIZERS AND COMPARE
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 1] NORMALIZER COMPARISON")
print("-" * 40)

from models.normalizer.normalizer import LinearNormalizer

# Load both normalizers
norm_official = LinearNormalizer.load(
    "/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"
)
norm_dataset = LinearNormalizer.load(
    "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt"
)

print("Official normalizer (umi_normalizer_official.pt):")
try:
    off_mean = norm_official['action'].mean
    off_std  = norm_official['action'].std
    print(f"  Mean: {np.round(np.array(off_mean), 4)}")
    print(f"  Std:  {np.round(np.array(off_std), 4)}")
except Exception as e:
    print(f"  Error reading: {e}")

print("\nDataset normalizer (normalizer.pt):")
try:
    ds_mean = norm_dataset['action'].mean
    ds_std  = norm_dataset['action'].std
    print(f"  Mean: {np.round(np.array(ds_mean), 4)}")
    print(f"  Std:  {np.round(np.array(ds_std), 4)}")
except Exception as e:
    print(f"  Error reading: {e}")

print("\n⚠️  If these are DIFFERENT → model was trained with one")
print("   but inference used the other → WRONG PREDICTIONS!")

# ──────────────────────────────────────────────────────────────
# FIX 2: CHECK INSTRUCTION FILE
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 2] INSTRUCTION FILE CHECK")
print("-" * 40)

instruction_path = os.path.join(SHARDS_DIR, 'instructions.json')

try:
    with open(instruction_path) as f:
        instructions = json.load(f)
    
    print(f"✅ Instructions loaded from: {instruction_path}")
    print(f"   Type: {type(instructions)}")
    
    if isinstance(instructions, dict):
        print(f"   Keys: {list(instructions.keys())[:5]}")
        # Show first few instructions
        for k, v in list(instructions.items())[:3]:
            print(f"   '{k}': '{v}'")
    elif isinstance(instructions, list):
        print(f"   Count: {len(instructions)}")
        print(f"   First 3: {instructions[:3]}")
    
    print("\n⚠️  IMPORTANT: Your inference instruction must EXACTLY match")
    print("   the instruction used during training!")
    print("   Training instruction ↑")
    print("   Inference instruction → 'Pick up the marker and place it in the box.'")
    
    if isinstance(instructions, dict):
        train_instructions = list(instructions.values())
    else:
        train_instructions = instructions
    
    INFERENCE_INSTRUCTION = "Pick up the marker and place it in the box."
    
    if any(INFERENCE_INSTRUCTION.lower() in str(i).lower() 
           for i in train_instructions[:10]):
        print("\n✅ Inference instruction matches training!")
    else:
        print("\n❌ MISMATCH! Training instructions are:")
        for i in train_instructions[:5]:
            print(f"   '{i}'")
        print(f"\n   But inference uses: '{INFERENCE_INSTRUCTION}'")
        print("   FIX: Change inference instruction to match training!")

except Exception as e:
    print(f"❌ Instructions error: {e}")
    import traceback
    traceback.print_exc()

# ──────────────────────────────────────────────────────────────
# FIX 3: VAE BOUNDS CHECK
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 3] VAE BOUNDS CHECK")
print("-" * 40)

try:
    from models.multivqvae import MultiVQVAE
    
    vae = MultiVQVAE.from_pretrained(
        "robotics-diffusion-transformer/RVQActionTokenizer"
    ).eval()
    
    print(f"✅ VAE loaded")
    print(f"   pos_id_len:  {vae.pos_id_len}")
    print(f"   rot_id_len:  {vae.rot_id_len}")
    print(f"   grip_id_len: {vae.grip_id_len}")
    
    # Check VAE internal bounds
    if hasattr(vae, 'pos_vq') or hasattr(vae, 'vq'):
        print("\n   VAE internal components:")
        for name, module in vae.named_modules():
            if 'vq' in name.lower() or 'codebook' in name.lower():
                print(f"   - {name}: {type(module).__name__}")
    
    # Test with real data
    root = zarr.open(DATASET_PATH, mode='r')
    pos  = np.array(root['data']['robot0_eef_pos'][:24])
    rot  = np.array(root['data']['robot0_eef_rot_axis_angle'][:24])
    grip = np.array(root['data']['robot0_gripper_width'][:24])
    
    # Fix the format error from before
    print(f"\n   First action:")
    print(f"   pos:  [{float(pos[0,0]):.4f}, {float(pos[0,1]):.4f}, {float(pos[0,2]):.4f}]")
    print(f"   rot:  [{float(rot[0,0]):.4f}, {float(rot[0,1]):.4f}, {float(rot[0,2]):.4f}]")
    print(f"   grip: {float(grip[0]):.4f}")
    
    # Check if normalizer brings actions to [-1, 1]
    print("\n   Checking if normalizer brings actions to proper range...")
    
    try:
        # Test normalization
        action_sample = np.concatenate([pos[0], rot[0], [float(grip[0])]])
        norm = norm_dataset  # Use dataset normalizer
        
        if hasattr(norm['action'], 'normalize'):
            normalized = norm['action'].normalize(
                torch.tensor(action_sample, dtype=torch.float32)
            )
            print(f"   Raw action:        {np.round(action_sample, 4)}")
            print(f"   Normalized action: {np.round(normalized.numpy(), 4)}")
            
            if normalized.abs().max() <= 1.5:
                print(f"   ✅ Normalization looks correct (max: {normalized.abs().max():.3f})")
            else:
                print(f"   ❌ Normalized values too large: {normalized.abs().max():.3f}")
                print(f"      Expected: max ~1.0")
                print(f"      FIX: Normalizer doesn't match dataset!")
                
    except Exception as e:
        print(f"   Normalization test error: {e}")

except Exception as e:
    print(f"❌ VAE test error: {e}")
    import traceback
    traceback.print_exc()

# ──────────────────────────────────────────────────────────────
# FIX 4: CREATE CORRECT NORMALIZER FROM DATASET
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 4] CREATE CORRECT NORMALIZER FROM DATASET")
print("-" * 40)

try:
    root   = zarr.open(DATASET_PATH, mode='r')
    pos    = np.array(root['data']['robot0_eef_pos'])
    rot    = np.array(root['data']['robot0_eef_rot_axis_angle'])
    grip   = np.array(root['data']['robot0_gripper_width'])
    
    # Compute stats from YOUR dataset
    actions     = np.concatenate([pos, rot, grip], axis=-1)
    action_mean = actions.mean(axis=0)
    action_std  = actions.std(axis=0)
    action_min  = actions.min(axis=0)
    action_max  = actions.max(axis=0)
    
    print("Dataset action statistics:")
    print(f"  Mean: {np.round(action_mean, 4)}")
    print(f"  Std:  {np.round(action_std, 4)}")
    print(f"  Min:  {np.round(action_min, 4)}")
    print(f"  Max:  {np.round(action_max, 4)}")
    
    # Save these for use in inference
    stats = {
        'mean': action_mean.tolist(),
        'std': action_std.tolist(),
        'min': action_min.tolist(),
        'max': action_max.tolist()
    }
    
    stats_path = "/home/rishabh/Downloads/umi-pipeline-training/dataset_action_stats.json"
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    
    print(f"\n✅ Dataset stats saved to: {stats_path}")
    print("   Use these for inference normalization!")

except Exception as e:
    print(f"❌ Stats error: {e}")

# ──────────────────────────────────────────────────────────────
# FIX 5: CHECK TRAIN LOG PROPERLY - FIND BEST TRAINING RUN
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 5] TRAINING HISTORY ANALYSIS")
print("-" * 40)

print("From train_resume.log:")
print("  First loss: 0.1767 @ lr=9e-07")
print("  Last loss:  0.0000 @ lr=9e-14")
print("  ❌ Loss went to 0 - LR decayed too fast!")
print("  ❌ Training happened at wrong point in schedule")

print("\nFrom train.log:")
print("  First loss: 8.3276 @ lr=9e-08")
print("  Last loss:  0.1836 @ lr=3e-12")
print("  ⚠️  Loss was still 0.18 at end - didn't converge")
print("  ⚠️  But this was learning properly (8.3 → 0.18)!")

print("\n📊 Training Analysis:")
print("  train.log:        8.32 → 0.18  (1 epoch, still learning)")
print("  train_resume.log: 0.18 → 0.00  (resumed, LR too small)")
print("\n  The model never fully converged!")
print("  Need: Fresh training with proper LR and more steps")

# ──────────────────────────────────────────────────────────────
# FINAL RECOMMENDATION
# ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print("🎯 ROOT CAUSE ANALYSIS & FIXES")
print("="*70)

print("""
ISSUES FOUND:
━━━━━━━━━━━━

1. 🚨 NORMALIZER MISMATCH (Most Critical)
   Training used:  normalizer.pt (your dataset normalizer)
   Inference used: umi_normalizer_official.pt (official)
   FIX: Use SAME normalizer for both training AND inference

2. 🚨 INSTRUCTION MISMATCH (Critical)
   Check if training instructions match:
   'Pick up the marker and place it in the box.'
   FIX: Use EXACT same instruction in inference

3. ❌ MODEL NEVER CONVERGED
   train.log:     loss=0.18 (not converged, only 1 epoch)
   train_resume:  loss=0.00 (LR=0, no learning)
   FIX: Need proper 3K-step training from scratch

4. ⚠️  VAE BOUNDS
   Check if your position values fit in RVQ codebook
   FIX: Verify normalizer brings all values to [-1, 1]

IMMEDIATE ACTION:
━━━━━━━━━━━━━━━

Step 1: Check instruction mismatch (5 mins)
Step 2: Try inference with correct normalizer (normalizer.pt)
Step 3: If still wrong → retrain properly
""")

print("="*70)
print("Share this output with your mentor!")
print("="*70)

🔧 FIXING ALL ISSUES

🔧 [FIX 1] NORMALIZER COMPARISON
----------------------------------------
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt
LinearNormalizer loaded from /home/rishabh/Downloads/umi-pipeline-training/normalizer.pt
Official normalizer (umi_normalizer_official.pt):
  Error reading: 'SingleFieldLinearNormalizer' object has no attribute 'mean'

Dataset normalizer (normalizer.pt):
  Error reading: 'ParameterDict' object has no attribute 'action'

⚠️  If these are DIFFERENT → model was trained with one
   but inference used the other → WRONG PREDICTIONS!

🔧 [FIX 2] INSTRUCTION FILE CHECK
----------------------------------------
✅ Instructions loaded from: /home/rishabh/Downloads/umi-pipeline-training/shards/instructions.json
   Type: <class 'dict'>
   Keys: ['task_0', 'task_1', 'task_2', 'task_3', 'task_4']
   'task_0': 'pick up the marker and place it in the box'
   'task_1': 'grasp the cup and move it to the table'
   '

/tmp/ipykernel_5445/740107734.py:140: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print(f"   grip: {float(grip[0]):.4f}")
/tmp/ipykernel_5445/740107734.py:147: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  action_sample = np.concatenate([pos[0], rot[0], [float(grip[0])]])


In [ ]:
import os, sys, json, torch, numpy as np
import zarr

RDT2_DIR     = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
DATASET_PATH = "/home/rishabh/Downloads/umi-pipeline-training/replay_buffer.zarr"
SHARDS_DIR   = "/home/rishabh/Downloads/umi-pipeline-training/shards"

sys.path.insert(0, RDT2_DIR)
sys.path.insert(0, os.path.join(RDT2_DIR, 'vqvae'))
sys.path.insert(0, os.path.join(RDT2_DIR, 'models'))
os.chdir(RDT2_DIR)

print("=" * 70)
print("🔧 FIXING ALL CRITICAL ISSUES")
print("=" * 70)

# ──────────────────────────────────────────────────────────────
# FIX 1: PRINT ALL INSTRUCTIONS IN TRAINING DATA
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 1] ALL TRAINING INSTRUCTIONS")
print("-" * 40)

instruction_path = os.path.join(SHARDS_DIR, 'instructions.json')
with open(instruction_path) as f:
    instructions = json.load(f)

print("All instructions in training data:")
for k, v in instructions.items():
    print(f"  {k}: '{v}'")

# Find which instruction matches pick and place
TARGET = "pick up the marker and place it in the box"
print(f"\n✅ CORRECT inference instruction to use:")
print(f"   '{TARGET}'")
print(f"   (lowercase, no period - must EXACTLY match training!)")

# ──────────────────────────────────────────────────────────────
# FIX 2: INSPECT NORMALIZER PROPERLY
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 2] NORMALIZER DEEP INSPECTION")
print("-" * 40)

# Inspect normalizer.pt (ParameterDict type)
print("Inspecting normalizer.pt:")
try:
    raw = torch.load(
        "/home/rishabh/Downloads/umi-pipeline-training/normalizer.pt",
        map_location='cpu'
    )
    print(f"  Type: {type(raw)}")
    
    if isinstance(raw, dict):
        print(f"  Keys: {list(raw.keys())}")
        for k, v in raw.items():
            if isinstance(v, torch.Tensor):
                print(f"  '{k}': tensor shape={v.shape}, "
                      f"min={v.min():.4f}, max={v.max():.4f}")
            elif isinstance(v, dict):
                print(f"  '{k}': dict with keys {list(v.keys())}")
                for k2, v2 in v.items():
                    if isinstance(v2, torch.Tensor):
                        print(f"    '{k2}': shape={v2.shape}")
    else:
        print(f"  Content: {raw}")
        
except Exception as e:
    print(f"  Error: {e}")

# Inspect umi_normalizer_official.pt
print("\nInspecting umi_normalizer_official.pt:")
try:
    from models.normalizer.normalizer import LinearNormalizer
    norm_official = LinearNormalizer.load(
        "/home/rishabh/Downloads/umi-pipeline-training/umi_normalizer_official.pt"
    )
    
    print(f"  Type: {type(norm_official)}")
    print(f"  Dir: {[x for x in dir(norm_official) if not x.startswith('_')]}")
    
    # Try all possible ways to access
    for key in ['action', 'obs', 'robot0_eef_pos']:
        try:
            val = norm_official[key]
            print(f"  norm['{key}']: {type(val)}")
            
            # Try to get stats
            for attr in ['mean', 'std', 'scale', 'offset', 
                        'input_min', 'input_max', 'params']:
                if hasattr(val, attr):
                    stat = getattr(val, attr)
                    if isinstance(stat, torch.Tensor):
                        print(f"    .{attr}: shape={stat.shape}, "
                              f"values={stat.flatten()[:4].tolist()}")
                    else:
                        print(f"    .{attr}: {stat}")
        except:
            pass
            
except Exception as e:
    print(f"  Error: {e}")
    import traceback
    traceback.print_exc()

# ──────────────────────────────────────────────────────────────
# FIX 3: COMPUTE CORRECT NORMALIZER FROM DATASET
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 3] COMPUTE CORRECT NORMALIZER FROM DATASET")
print("-" * 40)

root   = zarr.open(DATASET_PATH, mode='r')
pos    = np.array(root['data']['robot0_eef_pos'])
rot    = np.array(root['data']['robot0_eef_rot_axis_angle'])
grip   = np.array(root['data']['robot0_gripper_width'])

# Combine all actions
actions = np.concatenate([pos, rot, grip.reshape(-1, 1)], axis=-1)

print(f"Dataset shape: {actions.shape}")
print(f"\nAction statistics (what normalizer SHOULD have):")
print(f"  Mean: {np.round(actions.mean(0), 4)}")
print(f"  Std:  {np.round(actions.std(0), 4)}")
print(f"  Min:  {np.round(actions.min(0), 4)}")
print(f"  Max:  {np.round(actions.max(0), 4)}")

# Check gripper shape
print(f"\nGripper shape: {grip.shape}")
print(f"Gripper sample: {grip[:5]}")

# ──────────────────────────────────────────────────────────────
# FIX 4: INSPECT SHARD FILES FOR TASK DISTRIBUTION
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 4] SHARD TASK DISTRIBUTION")
print("-" * 40)

try:
    # Count shards per task
    shard_files = sorted([
        f for f in os.listdir(SHARDS_DIR) 
        if f.endswith('.tar')
    ])
    
    print(f"Total shards: {len(shard_files)}")
    
    # Check meta files for task distribution
    meta_files = [f for f in os.listdir(SHARDS_DIR) if f.endswith('.json')]
    print(f"Meta files: {meta_files}")
    
    for meta_f in meta_files:
        meta_path = os.path.join(SHARDS_DIR, meta_f)
        try:
            with open(meta_path) as f:
                meta = json.load(f)
            print(f"\n{meta_f}:")
            if isinstance(meta, dict):
                for k, v in list(meta.items())[:10]:
                    print(f"  {k}: {v}")
            elif isinstance(meta, list):
                print(f"  Count: {len(meta)}")
                print(f"  Sample: {meta[:3]}")
        except:
            pass
            
except Exception as e:
    print(f"  Shard check error: {e}")

# ──────────────────────────────────────────────────────────────
# FIX 5: TEST INFERENCE WITH CORRECT INSTRUCTION
# ──────────────────────────────────────────────────────────────
print("\n🔧 [FIX 5] CORRECT INFERENCE INSTRUCTION")
print("-" * 40)

CORRECT_INSTRUCTION = "pick up the marker and place it in the box"

print(f"❌ Wrong (was using): 'Pick up the marker and place it in the box.'")
print(f"✅ Correct:           '{CORRECT_INSTRUCTION}'")
print(f"\nThis EXACT string must be used in inference!")

# ──────────────────────────────────────────────────────────────
# FINAL SUMMARY & ACTION PLAN
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("🎯 FINAL ROOT CAUSE & ACTION PLAN")
print("=" * 70)

print("""
ROOT CAUSES CONFIRMED:
━━━━━━━━━━━━━━━━━━━━━

1. 🚨 INSTRUCTION MISMATCH (Easy fix - 5 mins)
   Training: 'pick up the marker and place it in the box'
   Inference: 'Pick up the marker and place it in the box.'
   FIX: Change inference instruction to EXACT training text

2. 🚨 NORMALIZER WRONG TYPE (Critical)
   normalizer.pt is ParameterDict (not LinearNormalizer)
   Training uses: normalizer.pt
   Inference uses: umi_normalizer_official.pt
   FIX: Use umi_normalizer_official.pt for BOTH training AND inference
   OR: Recompute normalizer from dataset correctly

3. 🚨 MIXED TASKS IN TRAINING (Confusing the model)
   Dataset has 5 different tasks:
   - pick up the marker and place it in the box ← WANT THIS
   - grasp the cup and move it to the table
   - fold the cloth neatly and place it on the shelf
   - (more tasks...)
   FIX: Filter dataset to ONLY marker task OR train longer

4. ❌ MODEL NEVER CONVERGED
   train.log: loss 8.3 → 0.18 (not converged)
   Need: Fresh 3K-step training with fixes above

ACTION PLAN (In order):
━━━━━━━━━━━━━━━━━━━━━━

STEP 1 (5 mins): Fix instruction in inference code
   Change to: 'pick up the marker and place it in the box'
   
STEP 2 (10 mins): Fix normalizer
   Use: umi_normalizer_official.pt for inference
   
STEP 3 (1 hour): Test if predictions improve
   Run inference with fixed instruction + normalizer
   
STEP 4 (if still bad): Retrain with 3K steps
   - Filter dataset to ONLY marker task
   - Use correct normalizer
   - Fresh training from scratch
""")

print("=" * 70)
print("⚡ QUICKEST FIX: Change instruction string in your inference code!")
print("=" * 70)


🔧 FIXING ALL CRITICAL ISSUES

🔧 [FIX 1] ALL TRAINING INSTRUCTIONS
----------------------------------------
All instructions in training data:
  task_0: 'pick up the marker and place it in the box'
  task_1: 'grasp the cup and move it to the table'
  task_2: 'fold the cloth neatly and place it on the shelf'
  task_3: 'open the drawer and retrieve the object inside'
  task_4: 'stack the blocks in a pyramid formation'

✅ CORRECT inference instruction to use:
   'pick up the marker and place it in the box'
   (lowercase, no period - must EXACTLY match training!)

🔧 [FIX 2] NORMALIZER DEEP INSPECTION
----------------------------------------
Inspecting normalizer.pt:
  Error: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set 

: 